Steps to preprocess MODIS data for phenology tracking. Note: GEE runs into memory issues when you try manipulating any of the big data for plotting/viewing/etc. So I am exporting these as geotiffs and reading them in as rasters to python to be able to look at the processing steps.

Author: Tayler LaSharr

Date: September 2024

In [29]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import os
import pprint

import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

In [30]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '11-01'
end_month_day = '03-30'
winterstart = 11
winterend = 3

## threshold for snow in winter to get the 'baseline' value in winter
snow_threshold = 50

## get the time period we care about
start_date_data = '2017-01-01'
end_date_data = '2023-12-31'


Read in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons)

In [31]:
## load in a shapefile or area of interest
## this is an assest that is already loaded into GEE

# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## dry af
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()

## load the MODIS NDVI data (before referencing its projection)
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA', 'State')

## function to calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add the NDVI band
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data so we can read in teh snow in the same projection
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()

## reproject the bounding box to the same projection as MODIS
reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounding box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## daily snow cover
modis_snow = ee.ImageCollection('MODIS/061/MOD10A1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDSI_Snow_Cover') \
                .map(lambda image: image.reproject(modis_projection).clip(reprojected_bounding_box))

## make sure the time property is there
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )
modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))

## just check on a couple of things
print('Projection of MODIS NDVI data:', modis_projection.getInfo())
print('Reprojected Bounding Box:', reprojected_bounding_box.getInfo())
print("Bands available in the first image:", modis_ndvi.first().bandNames().getInfo())


Projection of MODIS NDVI data: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}
Reprojected Bounding Box: {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-9195643.896612, 4639491.792059001], [-9017206.931857, 4639491.792059001], [-8957782.74624, 4686382.625733], [-8897873.318735, 4733273.459407], [-8837481.894627, 4780164.293081], [-8776611.745312, 4827055.126755999], [-8950287.693141999, 4827055.126755999], [-9012362.371173, 4780164.293081], [-9073948.850745, 4733273.459407], [-9135043.795726, 4686382.625733], [-9195643.896612, 4639491.792059001]]]}
Bands available in the first image: ['sur_refl_b01', 'sur_refl_b02', 'QA', 'State', 'NDVI']


In [32]:
## this step exports a bounding box for your study area. This will eventually be unnecessary, I was using it
## to crop the rasters to to get rid of weird values that were happening outside of the study area
## so I could look at the trends through time. Can comment this out if unneeded.

# bounding_box_fc = ee.FeatureCollection(ee.Feature(bounding_box))

# task = ee.batch.Export.table.toDrive(
#     collection=bounding_box_fc,
#     description='BoundingBoxExport',
#     folder='DroughtSurfing',  ## Google Drive folder where you want to save the shapefile
#     fileNamePrefix='bounding_box',  ## shapefile name
#     fileFormat='SHP'  ## format for exporting
# )

# ## start the export task
# task.start()

print('Bounding box export to Google Drive')

Bounding box export to Google Drive


Step 1. Remove low-quality observations with the QA band.

In [33]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow

def quality_and_state_mask(image, allow_snow=True):

    ## get both the state and quality bands
    state = image.select('State')
    quality = image.select('QA')

    ## state band masks
    good_cloud_state = state.bitwiseAnd(3).lt(2)  # Clear or Cloudy, exclude Mixed and Not set
    no_cloud_shadow = state.bitwiseAnd(1 << 2).eq(0)  # No Cloud shadow
    is_land = state.rightShift(3).bitwiseAnd(7).eq(1)  # Land
    no_snow_ice = state.bitwiseAnd(1 << 12).eq(0)  # No snow/ice

    ## quality band masks
    good_quality = quality.bitwiseAnd(3).eq(0)  # Highest quality

    ## combine masks: apply all conditions
    combined_mask = good_cloud_state.And(no_cloud_shadow).And(is_land).And(good_quality)

    if not allow_snow:
        combined_mask = combined_mask.And(no_snow_ice)

    ## apply the mask to the NDVI band
    ndvi_masked = image.updateMask(combined_mask).select('NDVI')

    return ndvi_masked

## apply the mask to the MODIS NDVI collection, assuming it contains 'state' and 'quality' bands correctly
modis_ndvi_filtered = modis_ndvi.map(lambda img: quality_and_state_mask(img, allow_snow=True))

## check the number of images (make sure it's running things and not getting rid of everything)
print("Number of images in the filtered collection:", modis_ndvi_filtered.size().getInfo())

print("Info:", modis_ndvi_filtered.first().getInfo())



Number of images in the filtered collection: 322
Info: {'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1810, 811], 'origin': [46704, 12762], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}], 'version': 1649921740560961, 'id': 'MODIS/061/MOD09Q1/2017_01_01', 'properties': {'system:footprint': {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-9195643.896612, 4639491.792059001], [-9017206.931857, 4639491.792059001], [-8957782.74624, 4686382.625733], [-8897873.318735, 4733273.459407], [-8837481.894627, 4780164.293081], [-8776611.745312, 4827055.126755999], [-8950287.693141999, 4827055.126755999], [-9012362.371173, 4780164.293081], [-9073948.850745, 4733273.459407], [-9135043.795726, 4686382.625733], [-9195643.896612, 4639491.792059001]]]}, 'system:time_start': 1483228800000, 'goo

Step 2. Set winter baselines dynamically since winter conditions are variable year to year. This differs from the Bischof and Merkle methods because winter is not consistent and some winters have snow while others do not. For this step we basically go through several different substeps. First, we define the winter period and add that attribute to the image collection. Then we calculate the 'baseline' winter values during that period for all cells that have less than a set amount of snow cover. Then we fill all empty cells during the winter period with the baseline value. Then, if there is snow covering the cells in winter, we will replace them with the winter baseline. Finally, to further smooth things in winter, if the image before and after the current image is set to the winter baseline, we will also set the current image to the baseline. This will prevent weird bumps from popping up throuhgout winter.

We don't set the winter values to the "baseline" for the entire winter because in desert systems there might be green that is more variable.

In [34]:
## get the baseline value to replace Null values in winter with
## need to make sure that this is doing what I want, something was happening where some of the 'baseline' values were -2000, so I added an
## argument that would not allow negative values to be considered when calculating the 0.025 quantile, but need to confirm that is not
## messing up how the baseline is calculated. check with Ellen on her thoughts here.

def calculate_baseline(image_collection, modis_snow, snow_threshold, quantile=0.025):
   
    def mask_snow_and_negatives(ndvi_image):
        
        ## get the date of the NDVI image
        image_date = ee.Date(ndvi_image.get('system:time_start'))

        ## create a buffer period around the NDVI image date (e.g., 16 days)
        buffer_period = 16
        snow_images = modis_snow.filterDate(image_date.advance(-buffer_period, 'day'), image_date.advance(buffer_period, 'day')).mean()

        ## create a snow mask where NDSI_Snow_Cover >= snow_threshold
        snow_mask = snow_images.select('NDSI_Snow_Cover').gte(snow_threshold)

        ## apply the snow mask to the NDVI image (exclude values where snow cover exceeds the threshold)
        return ndvi_image.updateMask(snow_mask.Not()).updateMask(ndvi_image.gte(0))

    ## apply the snow mask to each image in the collection
    snow_and_negatives_masked = image_collection.map(mask_snow_and_negatives)

    ## calculate the number of images used to get the baseline
    image_count = snow_and_negatives_masked.select('NDVI').reduce(ee.Reducer.count()).rename('image_count')

    ## calculate the 0.025 quantile (or any specified quantile) for NDVI, excluding snow-covered pixels
    baseline_ndvi = snow_and_negatives_masked.select('NDVI').reduce(ee.Reducer.percentile([quantile * 100])).rename('NDVI_baseline')

    return baseline_ndvi.addBands(image_count)

## calculate the baseline for the entire collection
winter_baseline_ndvi = calculate_baseline(modis_ndvi_filtered, modis_snow, snow_threshold=snow_threshold)
print("Winter Baseline NDVI calculated.")

print("Info:", winter_baseline_ndvi.getInfo())

Winter Baseline NDVI calculated.
Info: {'type': 'Image', 'bands': [{'id': 'NDVI_baseline', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'image_count', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 4294967295}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]}


In [35]:

## generate a mask that covers all cells with snow cover greater than threshold we defined
def generate_snow_mask(ndvi_image, modis_snow, snow_threshold):
    image_date = ee.Date(ndvi_image.get('system:time_start'))
    buffer_period_pre = 14  # two weeks prior
    buffer_period_post = 7  # one week after
    snow_images = modis_snow.filterDate(image_date.advance(-buffer_period_pre, 'day'), image_date.advance(buffer_period_post, 'day')).mean()
    snow_mask = snow_images.select('NDSI_Snow_Cover').gte(snow_threshold)
    return snow_mask

## function to add the snow mask to the NDVI image
def add_snow_mask(ndvi_image):
    snow_mask = generate_snow_mask(ndvi_image, modis_snow, snow_threshold)
    return ndvi_image.addBands(snow_mask.rename('snow_mask'))


## function to add a winter mask to the image
def add_winter_mask(image):
    month = ee.Date(image.get('system:time_start')).get('month')
    winter_mask = ee.Number(month).gte(winterstart).Or(ee.Number(month).lte(winterend))
    return image.addBands(ee.Image(winter_mask).rename('winter_mask'))

## apply the winter mask to the collection
modis_ndvi_with_winter_mask = modis_ndvi.map(add_winter_mask)

## function to apply the correction
def apply_corrections_with_snow(image):
    winter_mask = image.select('winter_mask')
    snow_mask = image.select('snow_mask')
    ndvi = image.select('NDVI')
    baseline_ndvi = winter_baseline_ndvi.select('NDVI_baseline')

    ## apply snow mask to NDVI
    masked_ndvi = ndvi.updateMask(snow_mask.Not())
    
    ## apply corrections only for winter months
    corrected_ndvi = masked_ndvi.where(winter_mask.eq(1).And(masked_ndvi.lt(baseline_ndvi)), baseline_ndvi)
    
    return image.addBands(corrected_ndvi.rename('NDVI'))

## apply the correction to the collection
modis_ndvi_with_snow_mask = modis_ndvi_with_winter_mask.map(add_snow_mask)
corrected_modis_ndvi = modis_ndvi_with_snow_mask.map(apply_corrections_with_snow)

print(corrected_modis_ndvi.first().getInfo())



{'type': 'Image', 'bands': [{'id': 'sur_refl_b01', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -32768, 'max': 32767}, 'dimensions': [1810, 811], 'origin': [46704, 12762], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}, {'id': 'sur_refl_b02', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -32768, 'max': 32767}, 'dimensions': [1810, 811], 'origin': [46704, 12762], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}, {'id': 'QA', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [1810, 811], 'origin': [46704, 12762], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}, {'id': 'State', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [1810, 811], 'origin': [4

In [36]:
def apply_neighbor_baseline_correction(ndvi_image, baseline_image, all_images, target_crs, target_scale):
    
    ## reproject the current NDVI image to the target CRS and scale
    ndvi_image = ndvi_image.reproject(crs=target_crs, scale=target_scale)

    ## reproject all images to the target CRS and scale
    all_images_list = all_images.toList(all_images.size())
    image_index = all_images_list.indexOf(ndvi_image)

    previous_image = ee.Image(ee.Algorithms.If(image_index.gt(0), all_images_list.get(image_index.subtract(1)), None))
    next_image = ee.Image(ee.Algorithms.If(image_index.lt(all_images.size().subtract(1)), all_images_list.get(image_index.add(1)), None))

    ## only reproject if the image is not None
    previous_image = ee.Algorithms.If(previous_image, ee.Image(previous_image).reproject(crs=target_crs, scale=target_scale), None)
    next_image = ee.Algorithms.If(next_image, ee.Image(next_image).reproject(crs=target_crs, scale=target_scale), None)

    def is_baseline(image):
        return image.select('NDVI').eq(baseline_image.select('NDVI_baseline')).reduceRegion(
            reducer=ee.Reducer.allNonZero(),
            geometry=image.geometry(),
            scale=target_scale
        ).values().get(0)

    prev_set_to_baseline = ee.Algorithms.If(previous_image, is_baseline(ee.Image(previous_image)), ee.Number(0))
    next_set_to_baseline = ee.Algorithms.If(next_image, is_baseline(ee.Image(next_image)), ee.Number(0))

    ## determine if both neighboring images are set to the baseline
    both_neighbors_set_to_baseline = ee.Number(prev_set_to_baseline).And(ee.Number(next_set_to_baseline))

    ## define corrections based on neighbors
    def correct_to_baseline():
        return baseline_image.select('NDVI_baseline').rename('NDVI')

    def keep_current_ndvi():
        return ndvi_image.select('NDVI')

    ## apply corrections based on neighbors
    corrected_ndvi = ee.Image(ee.Algorithms.If(both_neighbors_set_to_baseline, correct_to_baseline(), keep_current_ndvi()))
    return ndvi_image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)


## apply_neighbor_check_to_collection function to run it for the entire image collection
def apply_neighbor_check_to_collection(image_collection, baseline_image, target_crs, target_scale):
    all_images = image_collection.sort('system:time_start')
    
    def check_neighbors(image):
        corrected_image = apply_neighbor_baseline_correction(image, baseline_image, all_images, target_crs, target_scale)
        return corrected_image.select(['NDVI']).rename('NDVI')
    corrected_collection = image_collection.map(check_neighbors)
    return corrected_collection


## set the target projection and scale from your baseline NDVI or the first image in your collection
target_projection = modis_ndvi.first().projection().getInfo()
target_crs = target_projection['crs']
target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

## apply the neighbor correction with consistent projections
modis_ndvi_corrected = apply_neighbor_check_to_collection(corrected_modis_ndvi, winter_baseline_ndvi, target_crs, target_scale)


Step 3. Apply a smoothing function (moving median window with a search window of 3). 

IMPORTANT: Jerod's steps include running the moving mdeian fitler BEFORE scaling and flooring the data to the baseline. But in both the Bischof and Merkle methods, it's written as happening after things are floored. I need to check in with him about this and why he did it that way instead of after.

In [37]:
def apply_image_median(image_collection, window_size=3):
    image_collection = image_collection.map(lambda img: img.select('NDVI').toFloat())  # Use toFloat instead of toShort

    def smooth_image(current_image):
        all_images = image_collection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())

        image_index = all_images_list.indexOf(current_image)
        half_window = ee.Number(window_size // 2)

        # Calculate the start and end indices
        start_index = ee.Number(image_index).subtract(half_window).max(0)
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        # Get the window of images
        window_images = all_images_list.slice(start_index, end_index.add(1))
        window_collection = ee.ImageCollection.fromImages(window_images)

        # Compute the median
        median_image = window_collection.median().toFloat()  # Keep NDVI in float

        # Return the median image with the original properties
        return median_image.copyProperties(current_image, current_image.propertyNames())

    # Apply the smoothing function to the collection
    smoothed_collection = image_collection.map(smooth_image)
    return smoothed_collection

# Apply the smoothing function to the corrected NDVI collection
smoothed_ndvi = apply_image_median(modis_ndvi_corrected, window_size=3)
smoothed_ndvi = smoothed_ndvi.map(lambda img: img.reproject(crs=target_crs, scale=target_scale))


Export the data (at each time step so we can plot and look at them)

In [44]:

## the raw data and the filtered data can be exported as a single stack
def reproject_and_export(image_collection, file_name, folder_name, target_crs, scale):
    try:

        ## reproject each image in the collection to the target CRS
        def reproject_image(image):
            return image.reproject(crs=target_crs, scale=scale)

        ## apply reprojection to the entire image collection
        reprojected_collection = image_collection.map(reproject_image)

        ## stack all reprojected images as bands in a single multi-band image
        stacked_image = reprojected_collection.toBands()

        ## get the projection of the reprojected images (just for logging purposes)
        reprojected_projection = reprojected_collection.first().projection()

        ## export the stacked image with the reprojected CRS and scale
        task = ee.batch.Export.image.toDrive(
            image=stacked_image,
            description=f'{file_name}_stack',
            folder=folder_name,  # Specify the folder in your Google Drive
            fileNamePrefix=file_name,  # Name for the exported file
            region=bounding_box,  # You can use the bounding box or full extent (omit for full extent)
            scale=scale,  # Set the scale to a reasonable value (e.g., 30 meters for WGS84)
            maxPixels=1e13,  # Avoid pixel limit issues by setting a high value
            fileFormat='GeoTIFF'  # Export as GeoTIFF
        )
        task.start()
        print(f'Exporting {file_name}_stack to Google Drive in {target_crs} with scale {scale}.')

    except Exception as e:
        print(f'Error exporting {file_name}: {e}')

## specify the target CRS and scale
target_crs = 'EPSG:4326'  # WGS84
scale = 250

## export using WGS84 as the target CRS
reproject_and_export(modis_ndvi.select('NDVI'), 'modis_ndvi', 'DroughtSurfing', target_crs, scale)
reproject_and_export(modis_ndvi_filtered.select('NDVI'), 'modis_ndvi_filtered', 'DroughtSurfing', target_crs, scale)

## takes about ~30 minutes to export the images for the Sublette herd area (but will fail in ~2 minutes if it is not working).
## Open up GEE (https://code.earthengine.google.com/) to see if it's working and it will give you errors if it doesn't work.


Exporting modis_ndvi_stack to Google Drive in EPSG:4326 with scale 250.
Exporting modis_ndvi_filtered_stack to Google Drive in EPSG:4326 with scale 250.


In [46]:
## function to export image collection by year and chunk

def export_collection_by_year(image_collection, file_name_prefix, start_year, end_year, folder_name, target_crs, scale, bounding_box):
    try:
        ## loop through each year and filter the collection by year
        for year in range(start_year, end_year + 1):
            ## filter the collection for the current year
            yearly_collection = image_collection.filterDate(f'{year}-01-01', f'{year}-12-31')

            ## check if the collection for the year is empty
            if yearly_collection.size().getInfo() > 0:

                ## reproject each image in the collection to the target crs
                def reproject_image(image):
                    return image.reproject(crs=target_crs, scale=scale)

                ## apply reprojection to the entire yearly collection
                reprojected_yearly_collection = yearly_collection.map(reproject_image)

                ## break the collection into smaller chunks if necessary
                chunk_size = 10  ## adjust chunk size as needed
                total_images = reprojected_yearly_collection.size().getInfo()

                for i in range(0, total_images, chunk_size):
                    chunk = reprojected_yearly_collection.toList(chunk_size, i)
                    stacked_image = ee.ImageCollection(chunk).toBands()

                    ## export the stacked image
                    task = ee.batch.Export.image.toDrive(
                        image=stacked_image,
                        description=f'{file_name_prefix}_{year}_chunk_{i // chunk_size}',
                        folder=folder_name,
                        fileNamePrefix=f'{file_name_prefix}_{year}_chunk_{i // chunk_size}',
                        region=bounding_box,
                        scale=scale,
                        maxPixels=1e13,
                        fileFormat='GeoTIFF'
                    )

                    task.start()
                    print(f'exporting {file_name_prefix}_{year}_chunk_{i // chunk_size} to google drive in {target_crs} with scale {scale}.')
            else:
                print(f"no images found for {year} in {file_name_prefix}.")

    except Exception as e:
        print(f'error exporting {file_name_prefix}: {e}')

## specify the target crs and scale
target_crs = 'EPSG:4326'  ## wgs84
scale = 250

## export the corrected and smoothed ndvi data by year
export_collection_by_year(corrected_modis_ndvi.select('NDVI'), 'corrected_modis_ndvi', 2017, 2023, 'DroughtSurfing', target_crs, scale, reprojected_bounding_box)
# export_collection_by_year(smoothed_ndvi.select('NDVI'), 'smoothed_ndvi', 2017, 2023, 'DroughtSurfing', target_crs, scale, reprojected_bounding_box)




exporting corrected_modis_ndvi_2017_chunk_0 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2017_chunk_1 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2017_chunk_2 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2017_chunk_3 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2017_chunk_4 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2018_chunk_0 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2018_chunk_1 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2018_chunk_2 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2018_chunk_3 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2018_chunk_4 to google drive in EPSG:4326 with scale 250.
exporting corrected_modis_ndvi_2019_chunk_0 to google drive in EPSG:4326 with scale 250.
exporting corrected_m

In [40]:

# # Get the first image (winter) and the twentieth image (non-winter) for visualization
# first_image = modis_ndvi_corrected.toList(modis_ndvi_corrected.size()).get(0)
# twentieth_image = modis_ndvi_corrected.toList(modis_ndvi_corrected.size()).get(19)
# first_image = ee.Image(first_image)
# twentieth_image = ee.Image(twentieth_image)

# first_image2 = smoothed_ndvi.toList(smoothed_ndvi.size()).get(0)
# twentieth_image2 = smoothed_ndvi.toList(smoothed_ndvi.size()).get(19)
# first_image2 = ee.Image(first_image2)
# twentieth_image2 = ee.Image(twentieth_image2)



# # Define visualization parameters
# vis_params = {
#     'min': 0,
#     'max': 1,
#     'palette': ['blue', 'white', 'green']  # Adding green for better NDVI differentiation
# }

# # Initialize the interactive map
# Map = geemap.Map()

# # Add the original NDVI layer to the map
# Map.addLayer(first_image.select('NDVI'), vis_params, 'NDVI Image 1')

# # Add the corrected NDVI layer for a non-winter month
# Map.addLayer(twentieth_image.select('NDVI'), vis_params, 'NDVI Image 20')

# # Add the original NDVI layer to the map
# Map.addLayer(first_image2.select('NDVI'), vis_params, 'Smoothed NDVI Image 1')

# # Add the corrected NDVI layer for a non-winter month
# Map.addLayer(twentieth_image2.select('NDVI'), vis_params, 'Smoothed NDVI Image 20')


# # Add the winter baseline NDVI
# Map.addLayer(winter_baseline_ndvi.select('NDVI_baseline'), vis_params, 'Winter Baseline NDVI')

# # Center the map on the bounding box
# Map.centerObject(reprojected_bounding_box)
# Map.addLayerControl()

# # Show the map
# Map


In [41]:
## export the winter baseline image

# def export_to_drive(image, file_name, folder_name, scale, region, description="Export image to Drive"):
#     ## Start an export task
#     task = ee.batch.Export.image.toDrive(
#         image=image,  # The image to export
#         description=description,  # A description for the export task
#         folder=folder_name,  # The Drive folder to which the image will be exported
#         fileNamePrefix=file_name,  # The prefix of the filename in Drive
#         scale=scale,  # The pixel resolution in meters
#         region=region,  # The region to export
#         fileFormat='GeoTIFF',  # The format of the file to export
#         maxPixels=1e13  # Increase this if you get an error about too many pixels
#     )
#     task.start()  # Start the export task
#     print(f"Export task {description} has been started. Check your Google Drive in the folder '{folder_name}'.")

# ## Define parameters
# file_name = 'winter_baseline_ndvi'
# folder_name = 'DroughtSurfing'
# scale = 500  # The scale in meters; adjust according to the resolution you need
# region = modis_ndvi_filtered.first().geometry()  # Define the region; adjust as necessary

# ## Export the image
# export_to_drive(winter_baseline_ndvi, file_name, folder_name, scale, region)

In [42]:
## CODE TO DO THINGS FROM PREVIOUS PROTOCOLS THAT I REMOVED FROM OUR STEPS

## this checks for snow in the previous 3 images to see if the NDVI data are real or not. This is methodology
## that Merkle develped to update the Bischof methodology. It helps deal with NDVI values that were being estimated
## in areas where snow still was covering the ground.

# def mask_snow_free_pixels(ndvi_collection):

#     ## convert the NDVI collection to a list of images
#     ndvi_list = ndvi_collection.toList(ndvi_collection.size())
    
#     def apply_mask_to_ndvi(index):
#         # Get the current NDVI image
#         ndvi_image = ee.Image(ndvi_list.get(index))

#         # Get the QA band from the current NDVI image
#         quality = ndvi_image.select('QA')

#         ## bit 14: Snow/Ice (0 = no snow/ice, 1 = snow/ice) for the current image
#         snow_current = quality.rightShift(14).bitwiseAnd(1).eq(0)  # 0 = snow-free

#         ## check if there are previous images in the list (i.e., ensure index is valid)
#         ## get the snow condition for the previous image (index - 1)
#         previous_1 = ee.Algorithms.If(ee.Number(index).gte(1), ndvi_list.get(ee.Number(index).subtract(1)), None)
#         snow_previous_1 = ee.Image(ee.Algorithms.If(previous_1, ee.Image(previous_1).select('QA').rightShift(14).bitwiseAnd(1).eq(0), ee.Image(1)))

#         ## get the snow condition for the image before the previous one (index - 2)
#         previous_2 = ee.Algorithms.If(ee.Number(index).gte(2), ndvi_list.get(ee.Number(index).subtract(2)), None)
#         snow_previous_2 = ee.Image(ee.Algorithms.If(previous_2, ee.Image(previous_2).select('QA').rightShift(14).bitwiseAnd(1).eq(0), ee.Image(1)))

#         ## create a snow-free mask: A pixel is snow-free if it's snow-free in all three time steps (current, previous_1, previous_2)
#         snow_free_mask = snow_current.And(snow_previous_1).And(snow_previous_2)

#         ## apply the snow-free mask to the NDVI image
#         return ndvi_image.updateMask(snow_free_mask)

#     ## map over the image indices and apply the mask
#     indices = ee.List.sequence(0, ndvi_collection.size().subtract(1))
#     return ee.ImageCollection(indices.map(apply_mask_to_ndvi))

# ## run the function to get the snow-free NDVI collection
# modis_ndvi_filtered = mask_snow_free_pixels(modis_ndvi_filtered)
